In [ ]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.5 MB/s eta 0:00:00


In [ ]:
!pip install tensorflow

In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, confusion_matrix
)

# Keras for LSTM
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout

# Custom modules
from GNN_Model import GAT, train as train_gat
from Standalone_Diffusion_Model import Diffusion, train as train_diffusion
from Diffusion_GAT_Model import DiffusionGAT, train as train_diffgat, compute_loss_only as compute_loss_diffgat
from Standalone_Transformer_Model import TransformerOnly, train_transformer, val_transformer
from Transformer_GAT_Model import TransformerGAT, train as train_transformer_gat, compute_loss as compute_loss_transformer_gat
from LSTM_Model import prepare_lstm_data, build_lstm_model
from negative_sampling import advanced_negative_sampling
from evaluate import compute_loss_only, evaluate


In [ ]:
# Choose model and dataset
selected_model = "LSTM"  # Options: "GAT", "Diffusion", "DiffusionGAT", "Transformer", "TransformerGAT"
dataset_path = "CallGraph_0.csv"  # Or "CallGraph_1.csv", etc.
time_window_size = 100
train_time_end = 7000
test_time_start = 7000
test_time_end = 10000
embedding_dim = 64
eval_fn = evaluate
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
# Load dataset
df = pd.read_csv(dataset_path, on_bad_lines='skip')
df['um'] = df['um'].astype(str)
df['dm'] = df['dm'].astype(str)
all_nodes = pd.concat([df['um'], df['dm']]).unique()
node_mapping = {node: idx for idx, node in enumerate(all_nodes)}
df['um_encoded'] = df['um'].map(node_mapping)
df['dm_encoded'] = df['dm'].map(node_mapping)

# Generate time windows
def create_time_windows(df, window_size, max_time):
    return [df[(df['timestamp'] >= start) & (df['timestamp'] < start + window_size)]
            for start in range(0, max_time, window_size)]

time_windows = create_time_windows(df, time_window_size, test_time_end)
train_windows = [w for w in time_windows if w['timestamp'].max() < train_time_end]
test_windows  = [w for w in time_windows if test_time_start <= w['timestamp'].min() < test_time_end]

# Graph creation
def create_graph(w):
    edge_index = torch.tensor([w['um_encoded'].values, w['dm_encoded'].values], dtype=torch.long)
    edge_attr = torch.tensor(w['timestamp'].values, dtype=torch.float).unsqueeze(-1)
    return Data(edge_index=edge_index, edge_attr=edge_attr)

train_graphs = [create_graph(w) for w in train_windows]
test_graphs  = [create_graph(w) for w in test_windows]
num_nodes = len(all_nodes)
embedding_layer = torch.nn.Embedding(num_nodes, embedding_dim).to(device)

for g in train_graphs + test_graphs:
    g.x = embedding_layer(torch.arange(num_nodes, device=device)).detach()

train_graphs = [g.to(device) for g in train_graphs]
test_graphs  = [g.to(device) for g in test_graphs]


In [ ]:
print(f"Running model: {selected_model}")
seeds = [0, 1, 2, 3, 4]
patience = 10
min_epochs = 30
smooth_window = 10
delta = 0.005

for seed in seeds:
    print(f"\n\n===== Seed {seed} =====")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)

    for fold, (tr_idx, val_idx) in enumerate(kf.split(train_graphs)):
        print(f"\n-- Fold {fold + 1} --")

        # ========== Model selection ==========
        if selected_model == "Diffusion":
            model = Diffusion(embedding_dim, 16).to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
            train_fn = train_diffusion
            loss_fn = compute_loss_only
            eval_fn = evaluate

        elif selected_model == "GAT":
            model = GAT(embedding_dim, 16).to(device)
            train_fn = train_gat
            loss_fn = compute_loss_only
            eval_fn = evaluate

        elif selected_model == "DiffusionGAT":
            model = DiffusionGAT(embedding_dim, 16).to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
            train_fn = train_diffgat
            loss_fn = compute_loss_diffgat
            eval_fn = evaluate

        elif selected_model == "Transformer":
            model = TransformerOnly(num_nodes, embedding_dim).to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
            train_step = train_transformer
            val_step = val_transformer

        elif selected_model == "TransformerGAT":
            model = TransformerGAT(embedding_dim, 16).to(device)
            train_fn = train_transformer_gat
            loss_fn = compute_loss_transformer_gat
            eval_fn = evaluate

        else:
            raise ValueError("Selected model not supported.")

        # ========== Training ==========
        val_losses = []
        best_smoothed_loss = float('inf')
        no_improve_epochs = 0
        best_state = None
        best_epoch = 0
        start_time = time.time()

        for epoch in range(1, 201):
            tr_loss = 0
            for i in tr_idx:
                g = train_graphs[i]
                g.x = embedding_layer(torch.arange(num_nodes, device=device)).detach()

                if selected_model == "Transformer":
                    tr_loss += train_step(model, g, optimizer, embedding_layer, num_nodes, device)
                else:
                    tr_loss += train_fn(model, g, optimizer)
            tr_loss /= len(tr_idx)

            val_loss = 0
            for i in val_idx:
                g = train_graphs[i]
                g.x = embedding_layer(torch.arange(num_nodes, device=device)).detach()

                if selected_model == "Transformer":
                    val_loss += val_step(model, g, embedding_layer, num_nodes, device)
                else:
                    val_loss += loss_fn(model, g)
            val_loss /= len(val_idx)

            val_losses.append(val_loss)
            smoothed = np.mean(val_losses[-smooth_window:]) if len(val_losses) >= smooth_window else np.mean(val_losses)
            print(f"Epoch {epoch:3d} — Train: {tr_loss:.4f} | Val: {val_loss:.4f} | Smoothed: {smoothed:.4f}")

            if epoch >= min_epochs:
                if smoothed < best_smoothed_loss * (1 - delta):
                    best_smoothed_loss = smoothed
                    best_state = {k: v.clone() for k, v in model.state_dict().items()}
                    best_epoch = epoch
                    no_improve_epochs = 0
                else:
                    no_improve_epochs += 1
                    if no_improve_epochs >= patience:
                        print(f"Early stopping at epoch {epoch}")
                        model.load_state_dict(best_state)
                        break

        train_duration = time.time() - start_time

        # ========== Evaluation ==========
        print(f"\n>>> Test results for seed {seed}, fold {fold + 1}")
        results = []
        eval_start = time.time()

        for i, tg in enumerate(test_graphs):
            tg.x = embedding_layer(torch.arange(num_nodes, device=device)).detach()
            auc, prec, rec, f1, acc, mrr = eval_fn(model, tg)
            print(f" Window {i:2d}: AUC={auc:.4f}, P={prec:.4f}, R={rec:.4f}, F1={f1:.4f}, Acc={acc:.4f}, MRR={mrr:.4f}")
            results.append({
                "Window": i, "AUC": round(auc, 4), "Precision": round(prec, 4),
                "Recall": round(rec, 4), "F1": round(f1, 4),
                "Accuracy": round(acc, 4), "MRR": round(mrr, 4)
            })

        eval_duration = time.time() - eval_start

        # ========== Save results ==========
        pd.DataFrame(results).to_csv(f"link_prediction_report_seed{seed}_fold{fold + 1}.csv", index=False)
        pd.DataFrame([{
            "Seed": seed,
            "Fold": fold + 1,
            "EarlyStopEpoch": best_epoch,
            "TrainTimeSec": round(train_duration, 2),
            "EvalTimeSec": round(eval_duration, 2),
            "NumTestWindows": len(test_graphs)
        }]).to_csv(f"metadata_seed{seed}_fold{fold + 1}.csv", index=False)
